In [20]:
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

In [21]:
with open('group_25_dataset.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

X = loaded_data['data']
y = loaded_data['target']

# Keep only V3 and I_E for the ML models
selected_cols = [2, 3]
X = X[:, :, selected_cols]
measurement = np.loadtxt('measurements.csv', delimiter=',')[:, selected_cols]

print(f"Input shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Measurement shape: {measurement.shape}")

Input shape: (2000, 500, 2)
Target shape: (2000, 2)
Measurement shape: (500, 2)


In [22]:
# Flatten the selected waveform channels into long feature vectors
X = X.reshape((X.shape[0], -1))

# Reshape measurement into one row
measurement = measurement.reshape(1, -1)

print(f"Input shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Measurement shape: {measurement.shape}")

Input shape: (2000, 1000)
Target shape: (2000, 2)
Measurement shape: (1, 1000)


In [23]:
from sklearn.model_selection import train_test_split

rng = np.random.seed(42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.5,
    random_state=42
)

print("Train set input shape:", X_train.shape)
print("Test set input shape:", X_test.shape)

Train set input shape: (1000, 1000)
Test set input shape: (1000, 1000)


In [24]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)

print("Linear Regression Results")
print("MSE:", mean_squared_error(y_test, y_pred_linear))
print("MAE:", mean_absolute_error(y_test, y_pred_linear))
print("R2:", r2_score(y_test, y_pred_linear))

Linear Regression Results
MSE: 9700202.350286938
MAE: 1740.458757048593
R2: -102.0089997586263


In [25]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Results")
print("MSE:", mean_squared_error(y_test, y_pred_rf))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("R2:", r2_score(y_test, y_pred_rf))

Random Forest Results
MSE: 3245.0236720522375
MAE: 11.625118892234548
R2: 0.7092370761509224


In [26]:
gbr_model = MultiOutputRegressor(
    GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
)

gbr_model.fit(X_train, y_train)

y_pred_gbr = gbr_model.predict(X_test)

print("Gradient Boosted Trees Results")
print("MSE:", mean_squared_error(y_test, y_pred_gbr))
print("MAE:", mean_absolute_error(y_test, y_pred_gbr))
print("R2:", r2_score(y_test, y_pred_gbr))

Gradient Boosted Trees Results
MSE: 2252.8746660312045
MAE: 11.001180340858605
R2: 0.8357471021959484


In [27]:
linear_pred = linear_model.predict(measurement)[0]
rf_pred = rf_model.predict(measurement)[0]
gbr_pred = gbr_model.predict(measurement)[0]

rows = [
    ["Linear Regression", f"{mean_squared_error(y_test, y_pred_linear):.6f}", f"{mean_absolute_error(y_test, y_pred_linear):.6f}", f"{r2_score(y_test, y_pred_linear):.6f}", f"{linear_pred[0]:.4f}", f"{linear_pred[1]:.4e}"],
    ["Random Forest", f"{mean_squared_error(y_test, y_pred_rf):.6f}", f"{mean_absolute_error(y_test, y_pred_rf):.6f}", f"{r2_score(y_test, y_pred_rf):.6f}", f"{rf_pred[0]:.4f}", f"{rf_pred[1]:.4e}"],
    ["Gradient Boosted Trees", f"{mean_squared_error(y_test, y_pred_gbr):.6f}", f"{mean_absolute_error(y_test, y_pred_gbr):.6f}", f"{r2_score(y_test, y_pred_gbr):.6f}", f"{gbr_pred[0]:.4f}", f"{gbr_pred[1]:.4e}"],
]

def print_table(title, headers, rows):
    widths = []
    for i, header in enumerate(headers):
        widths.append(max(len(header), *(len(row[i]) for row in rows)))

    print(title)
    print(" | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))))
    print("-+-".join("-" * widths[i] for i in range(len(headers))))
    for row in rows:
        print(" | ".join(row[i].ljust(widths[i]) for i in range(len(headers))))

print_table(
    "Model Summary for measurements.csv",
    ["Method", "MSE", "MAE", "R2", "R_pred (ohms)", "C_pred (F)"],
    rows,
)

Model Summary for measurements.csv
Method                 | MSE            | MAE         | R2          | R_pred (ohms) | C_pred (F) 
-----------------------+----------------+-------------+-------------+---------------+------------
Linear Regression      | 9700202.350287 | 1740.458757 | -102.009000 | -4388.5065    | -1.2523e-05
Random Forest          | 3245.023672    | 11.625119   | 0.709237    | 1027.3855     | 1.2050e-06 
Gradient Boosted Trees | 2252.874666    | 11.001180   | 0.835747    | 988.8775      | 1.4015e-06 
